In [ ]:
import importlib
import sys
from pathlib import Path

import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

workspace_root = Path.cwd()
if not (workspace_root / "compression_knn").exists():
    workspace_root = workspace_root.parent
sys.path.insert(0, str(workspace_root))

import compression_knn.knn as knn_module
import compression_knn.preprocessor as preprocessor_module

importlib.reload(knn_module)
importlib.reload(preprocessor_module)

from compression_knn.knn import CompressionKNNClassifier
from compression_knn.knn import CompressionKNNClassifierCV
from compression_knn.preprocessor import VectorToTextTransformer

print(f"workspace_root={workspace_root}")

# Vector Inputs, Pipelines, and CV

This notebook tests the non-text workflow in the repo: turning vectors into text and then classifying them with the same compression-based estimator.

In [ ]:
X_train_num = np.array([
    [1, 1],
    [1, 2],
    [9, 9],
    [8, 9],
], dtype=float)
y_train_num = np.array(["low", "low", "high", "high"], dtype=str)

X_test_num = np.array([
    [1, 0],
    [9, 8],
], dtype=float)
y_test_num = np.array(["low", "high"], dtype=str)

transformer = VectorToTextTransformer(separator=",")
text_train_preview = transformer.fit_transform(X_train_num)

print("text_train_preview=", text_train_preview.tolist())

In [ ]:
pipeline = Pipeline(
    [
        ("vector_to_text", VectorToTextTransformer(separator=",")),
        ("classifier", CompressionKNNClassifier(n_neighbors=1, random_state=0)),
    ]
)
pipeline.fit(X_train_num, y_train_num)
pipeline_predictions = pipeline.predict(X_test_num)

print("pipeline_predictions=", pipeline_predictions.tolist())
assert np.array_equal(pipeline_predictions, y_test_num)

## Example 2: Mixed-Type Rows Through the Transformer

In [ ]:
mixed_rows = np.array(
    [[1, 2.5, True], ["red", None, "x"]],
    dtype=object,
)
mixed_transformer = VectorToTextTransformer(separator="|")
mixed_as_text = mixed_transformer.fit_transform(mixed_rows)

print("mixed_as_text=", mixed_as_text.tolist())
assert mixed_as_text.tolist() == ["1|2.5|True", "red|None|x"]

## Example 3: Cross-Validate the Numeric Pipeline

In [ ]:
X_all_num = np.vstack([X_train_num, X_test_num])
y_all_num = np.concatenate([y_train_num, y_test_num])

cv_pipeline = Pipeline(
    [
        ("vector_to_text", VectorToTextTransformer(separator=",")),
        ("classifier", CompressionKNNClassifier(n_neighbors=1, random_state=0)),
    ]
)
cv_scores = cross_val_score(cv_pipeline, X_all_num, y_all_num, cv=3, scoring="accuracy")

print("cv_scores=", cv_scores.tolist())
print(f"mean_cv_accuracy={cv_scores.mean():.3f}")

In [ ]:
textified_rows = VectorToTextTransformer(separator=",").fit_transform(X_all_num)
cv_classifier = CompressionKNNClassifierCV(
    n_neighbors=[1, 3],
    compressor="gzip",
    cv=3,
    random_state=0,
)
cv_classifier.fit(textified_rows, y_all_num)

print(f"selected_n_neighbors={cv_classifier.n_neighbors_}")
print(f"best_score={cv_classifier.best_score_:.3f}")
print("cv_result_=")
print(np.round(cv_classifier.cv_result_, 3))

## Example 4: Predict New Numeric Points

In [ ]:
new_points = np.array([
    [1, 1.5],
    [8.5, 9.0],
], dtype=float)
new_predictions = pipeline.predict(new_points)

print("new_predictions=", new_predictions.tolist())
assert set(new_predictions.tolist()) == {"low", "high"}

These examples verify that the transformer, pipeline, and cross-validated classifier all work on small numeric toy datasets after converting rows into text.